# Домашнее задание 6

В этом задании мы:
1. Посмотрим на то, как получать эмбеддинги слов с помощью счетчиков и с помощью word2vec.
2. Построим простую сеть для классификации текста, замерим качество.
3. Посмотрим, как меняется качество этой сети в зависимости от качества эмбеддингов.
4. Построим TextCNN, обучим, сравним качество.

Работать будем с датасетом IMDb.

In [1]:
# Загрузка данных IMDb с помощью Keras
from keras.datasets import imdb

# Датасет хранится в двух кусках: сами данные и индексы (трейн/тест)
# Скачаем индексы
(X_train_indices, y_train), (X_test_indices, y_test) = imdb.load_data(num_words=10000)
# И сами данные. Это словарь вида {"слово": 0, "другое": 100}
# Выделим первые три слота под специальные токены: <PAD>, <START> и <UNK>
# Нумерация в исходном датасете идет с единицы, поэтому вычтем 1
word_to_index = {k: v - 1 + 3 for k, v in imdb.get_word_index().items()} | {
    "<PAD>": 0,
    "<START>": 1,
    "<UNK>": 2,
}
index_to_word = {index: word for word, index in word_to_index.items()}


# Функция для преобразования последовательности индексов в последовательность слов
def indices_to_words(indices):
    return " ".join(index_to_word.get(index, "<UNK>") for index in indices)


# Список строк, вида ["Good movie", "Bad film", ...]
X_train = [indices_to_words(indices) for indices in X_train_indices]
X_test = [indices_to_words(indices) for indices in X_test_indices]

2026-03-11 09:41:36.055958: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773222096.376500      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773222096.472218      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773222097.278822      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773222097.278872      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773222097.278874      55 computation_placer.cc:177] computation placer alr

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


В отзывах на фильмы скорее всего будут использовать различные сокращения фраз.
Например, AFAIK - "as far as I know".
Мы собрали словарь некоторых из них.

In [2]:
chat_words = {
    "AFAIK": "As Far As I Know",
    "AFK": "Away From Keyboard",
    "ASAP": "As Soon As Possible",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "A3": "Anytime, Anywhere, Anyplace",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRT": "Be Right There",
    "BTW": "By The Way",
    "B4": "Before",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FWIW": "For What It's Worth",
    "FYI": "For Your Information",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GN": "Good Night",
    "GMTA": "Great Minds Think Alike",
    "GR8": "Great!",
    "G9": "Genius",
    "IC": "I See",
    "ICQ": "I Seek you (also a chat program)",
    "ILU": "I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "LDR": "Long Distance Relationship",
    "LTNS": "Long Time No See",
    "L8R": "Later",
    "MTE": "My Thoughts Exactly",
    "M8": "Mate",
    "NRN": "No Reply Necessary",
    "OIC": "Oh I See",
    "PITA": "Pain In The A..",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "QPSA": "Que Pasa?",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A.. Off",
    "SK8": "Skate",
    "STATS": "Your sex and age",
    "ASL": "Age, Sex, Location",
    "THX": "Thank You",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "WB": "Welcome Back",
    "WTF": "What The F...",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "W8": "Wait...",
    "7K": "Sick:-D Laughter",
    "TFW": "That feeling when",
    "MFW": "My face when",
    "MRW": "My reaction when",
    "IFYP": "I feel your pain",
    "LOL": "Laughing out loud",
    "TNTL": "Trying not to laugh",
    "JK": "Just kidding",
    "IDC": "I don’t care",
    "ILY": "I love you",
    "IMU": "I miss you",
    "ADIH": "Another day in hell",
    "ZZZ": "Sleeping, bored, tired",
    "WYWH": "Wish you were here",
    "BAE": "Before anyone else",
    "FIMH": "Forever in my heart",
    "BSAAW": "Big smile and a wink",
    "BWL": "Bursting with laughter",
    "LMAO": "Laughing my a** off",
    "BFF": "Best friends forever",
    "CSL": "Can’t stop laughing",
}
# Чтобы не иметь проблем с регистром, приведем все к нижнему регистру
chat_words = {key.lower(): value.lower() for key, value in chat_words.items()}

### Задание №1
Посчитайте, какое количество таких сокращений встречается в тестовом наборе данных `X_train`.

Например, для строк:
```
[
    "AFAIK, he is AFK",
    "Need to go, BBS",
    "Are you AFK ?",
]
```
ответ будет 4, т.к. всего в текстах встретилось 4 сокращения: AFAIK, AFK, BBS, AFK.

Сдайте в ЛМС одно число - количество сокращений. Например, `123`.

In [ ]:
type(X_train_indices)

In [ ]:
indices_to_words(X_train_indices[0])

In [ ]:
chat_words.keys()

In [ ]:
y_train.shape

In [ ]:
len(index_to_word.keys())

In [3]:
from collections import Counter
import numpy as np

flat_words = [index_to_word.get(index) for index in np.concatenate(X_train_indices)]
print(len(flat_words))

# 1. Считаем частоту всех слов во втором списке
target_counter = Counter(flat_words)
# print(target_counter)

# 2. Создаем словарь с результатами
result = {}
for word in chat_words.keys():
    result[word] = target_counter.get(word, 0) # get вернет 0, если слова нет

print(result)
print(sum(result.values()))

5967841
{'afaik': 0, 'afk': 0, 'asap': 0, 'atk': 0, 'atm': 0, 'a3': 0, 'bak': 0, 'bbl': 0, 'bbs': 0, 'bfn': 0, 'b4n': 0, 'brb': 0, 'brt': 0, 'btw': 65, 'b4': 0, 'cu': 0, 'cul8r': 0, 'cya': 0, 'faq': 0, 'fc': 0, 'fwiw': 0, 'fyi': 0, 'gal': 71, 'gg': 0, 'gn': 0, 'gmta': 0, 'gr8': 0, 'g9': 0, 'ic': 0, 'icq': 0, 'ilu': 0, 'imho': 44, 'imo': 59, 'iow': 0, 'irl': 0, 'ldr': 0, 'ltns': 0, 'l8r': 0, 'mte': 0, 'm8': 0, 'nrn': 0, 'oic': 0, 'pita': 32, 'prt': 0, 'prw': 0, 'qpsa': 0, 'rofl': 0, 'roflol': 0, 'rotflmao': 0, 'sk8': 0, 'stats': 0, 'asl': 0, 'thx': 0, 'ttfn': 0, 'ttyl': 0, 'u2': 0, 'u4e': 0, 'wb': 31, 'wtf': 64, 'wtg': 0, 'wuf': 0, 'w8': 0, '7k': 0, 'tfw': 0, 'mfw': 0, 'mrw': 0, 'ifyp': 0, 'lol': 109, 'tntl': 0, 'jk': 0, 'idc': 0, 'ily': 0, 'imu': 0, 'adih': 0, 'zzz': 0, 'wywh': 0, 'bae': 0, 'fimh': 0, 'bsaaw': 0, 'bwl': 0, 'lmao': 0, 'bff': 0, 'csl': 0}
475


In [3]:
# решение авторов
import tqdm

cnt = 0
for text in tqdm.tqdm(X_train):
    for word in text.split():
        if word in chat_words:
            cnt += 1
cnt

100%|██████████| 25000/25000 [00:00<00:00, 27336.41it/s]


475

Подумайте самостоятельно: стоит ли заменять эти сокращения в `X_train` и `X_test`?

## Простая модель, простой эмбеддинг
Обучим на этих данных классификатор.
Для этого надо слова превратить в вектора - _эмбеддинги_, затем построить модель.

### Задание №2

Обучите `CountVectorizer` из scikit-learn на текстах - это будет модель для эмбеддингов.
Используйте `max_features=10000`.

Затем обучите простую нейросеть (код ниже) на классификацию отзывов.

Сдайте в ЛМС код модели `SimpleNeuralNet` и файл с весами обученной модели (`model.pt`).

In [ ]:
X_train[-1]

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=2)
X = vectorizer.fit_transform(X_train)
print(vectorizer.get_feature_names_out())
print(X.toarray())

['and' 'unk']
[[15  6]
 [15  7]
 [ 9  2]
 ...
 [13 14]
 [ 5  5]
 [10 12]]


In [ ]:
print(type(X))

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.feature_extraction.text import CountVectorizer
import tqdm


class SimpleNeuralNet(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        out = self.fc(x)
        return out.squeeze()


vectorizer = CountVectorizer(max_features=10000)
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

model = SimpleNeuralNet(input_dim=X_train_bow.shape[1])

In [20]:
class BowDataset(Dataset):
    def __init__(self, texts, labels):
        self.data = texts
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        label = torch.tensor(self.labels[index], dtype=torch.float32)
        word_matrix = torch.tensor(
            self.data[index].toarray(),
            dtype=torch.float32
        ).squeeze()

        return word_matrix, label

train_ds = BowDataset(X_train_bow, y_train)
test_ds = BowDataset(X_test_bow, y_test)

train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    # num_workers=4
)

test_loader = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False,
    # num_workers=4
)

In [26]:
# TensorDataset
X_train_dense = torch.tensor(X_train_bow.toarray(), dtype=torch.float32)
X_test_dense = torch.tensor(X_test_bow.toarray(), dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

train_ds = TensorDataset(X_train_dense, y_train_t)
test_ds = TensorDataset(X_test_dense, y_test_t)

train_loader = DataLoader(
    train_ds,
    batch_size=512,
    shuffle=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=512
)

In [28]:
@torch.no_grad()
def loss_on_dataset(model: nn.Module, loader: DataLoader, device: torch.device):
    model.to(device).eval()
    loss = 0
    for x, label in loader:
        pred = model(x)
        loss = loss + criterion(pred, label)
    model.train()
    return (loss / len(loader)).item()


@torch.no_grad()
def accuracy(model: nn.Module, loader: DataLoader, device: torch.device):
    count_correct, count_total = 0, 0
    model.to(device).eval()
    for x, labels in loader:
        x = x.to(device)
        labels = labels.to(device).squeeze()
        pred_val = model(x).squeeze()
    
        pred_labels = (pred_val >= 0.5).float()
        count_correct += (pred_labels == labels).sum().item()
        count_total += len(labels)
        
    model.train()
    return count_correct / count_total

acc = []
criterion = nn.BCELoss()

def train_loop(model: nn.Module, 
               train_loader: DataLoader, 
               test_loader: DataLoader, 
               device: torch.device, 
               n_epochs: int):
    
    optimizer = Adam(model.parameters(), lr=1e-4)
    model.to(device)
    
    for epoch in range(n_epochs):
        print(f"Epoch {epoch}")
        model.train()
        for i, (X, label) in enumerate(tqdm.tqdm(train_loader)):
            optimizer.zero_grad()
            X = X.to(device)
            
            label = label.to(device).squeeze()
            pred = model(X)
            
            # print(pred.dtype, label.dtype)
            loss = criterion(pred, label)
            loss.backward()
            optimizer.step()
            if i % 10 == 0:
                accu = accuracy(model, test_loader, device)
                acc.append(accu)
                print(f"Итерация {i}: точность {accu}")
        

In [32]:
device = "cuda" if torch.cuda.is_available() else "cpu" 
# accuracy(model, test_loader, device)
# model = SimpleNeuralNet(input_dim=X_train_bow.shape[1])
train_loop(model, train_loader, test_loader, device, n_epochs=10)

Epoch 0


 16%|█▋        | 8/49 [00:00<00:02, 14.01it/s]

Итерация 0: точность 0.86908


 41%|████      | 20/49 [00:01<00:01, 16.05it/s]

Итерация 10: точность 0.86932


...

Итерация 20: точность 0.87276


 67%|██████▋   | 33/49 [00:03<00:01, 10.67it/s]

Итерация 30: точность 0.87288


100%|██████████| 49/49 [00:03<00:00, 12.33it/s]

Итерация 40: точность 0.8742


In [33]:
torch.save(model.state_dict(), "model.pt")

In [34]:
# Решение авторов
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.feature_extraction.text import CountVectorizer  # isort: skip
from sklearn.metrics import classification_report


class SimpleNeuralNet(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        out = self.fc(x)
        return out.squeeze()


vectorizer = CountVectorizer(max_features=10_000)
X_train_bow = vectorizer.fit_transform(X_train).toarray()
X_test_bow = vectorizer.transform(X_test).toarray()


def train_neural_net(X_train, y_train, X_test, y_test, input_dim):
    model = SimpleNeuralNet(input_dim)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Преобразуем данные в тензоры PyTorch
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

    train_data = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

    # Обучение модели
    for epoch in range(50):
        model.train()
        for inputs, labels in train_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Тестирование модели
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        preds = (outputs > 0.5).float()
        print("\nПростая нейронная сеть:")
        print(classification_report(y_test_tensor, preds))
    return model


model = train_neural_net(X_train_bow, y_train, X_test_bow, y_test, X_train_bow.shape[1])
# torch.save(model.state_dict(), "model.pt")


Простая нейронная сеть:
              precision    recall  f1-score   support

         0.0       0.85      0.87      0.86     12500
         1.0       0.86      0.85      0.85     12500

    accuracy                           0.86     25000
   macro avg       0.86      0.86      0.86     25000
weighted avg       0.86      0.86      0.86     25000



In [ ]:
# Почистим память перед дальнейшими экспериментами
del X_train_bow, X_test_bow, vectorizer

## Word2Vec

Попробуем использовать более продвинутую модель для подсчета эмбеддингов.
Возьмем `word2vec` - это популярная модель для эмбеддингов, ее часто используют в качестве бейзлайна.

In [ ]:
from gensim.downloader import load

word2vec = load("word2vec-google-news-300")
word2vec.similar_by_word("cat")

Про word2vec часто говорят, что он дает "осмысленные" эмбеддинги.
Давайте это проверим на примере.

### Задание №3
Какое слово получается, если сделать Paris - France + Germany, и с какой косинусной близостью?

Сдайте в ЛМС слово и близость в формате `'Munich', 0.98`.

Можно надеяться, что если эмбеддинги будут более осмысленные, то качество модели увеличится.
Давайте проверять.

### Задание №4

Возьмите ту же модель `SimpleNeuralNet` и обучите её, взяв эмбеддинги из `word2vec`.

Вы, возможно, подумаете: `word2vec` выдает по вектору на каждое слово, а в предложении слов много.
Как с этим быть, предлагаем подумать самостоятельно.

Сдайте в ЛМС код `SimpleNeuralNet` и файл с весами модели `model.pt`.

In [ ]:
import numpy as np


def sentence_to_embedding(sentence: str) -> np.ndarray:
    # На каждое предложение возвращаем 1 вектор размерности (300, )
    # Подумайте, что делать со словами, которых нет в словаре word2vec.
    ...


X_train_w2v = np.stack(
    [sentence_to_embedding(sentence) for sentence in tqdm.tqdm(X_train)]
)
X_test_w2v = np.stack(
    [sentence_to_embedding(sentence) for sentence in tqdm.tqdm(X_test)]
)
assert X_train_w2v.shape == (25_000, 300)
assert X_test_w2v.shape == (25_000, 300)

In [ ]:
...
torch.save(model.state_dict(), "model-w2v.pt")

In [ ]:
del X_train_w2v, X_test_w2v

Возможно, качество вырастет не так сильно, но зато теперь мы можем считать эмбеддинги отдельно для каждого слова.
А это значит, что мы можем попробовать учесть порядок слов в предложении.

## TextCNN

Модель TextCNN - одна из первых попыток учесть положение слов в тексте.
Попробуем обучить эту модель.

### Задание №5

Обучите `TextCNN` на датасете IMDb, используя эмбеддинги `word2vec` и паддинг из прошлого пункта.

Не забудьте, что в таком подходе каждое предложение имеет размерность не `(300, )`, а `(n_words, 300)`.
А это значит:
- нужен паддинг;
- нужно переписать логику подсчета эмбеддингов для предложения.

Вы можете поискать подсказки в семинаре.

Сдайте в ЛМС код `TextCNN` и файл с весами модели.

### Задание №6
Поэкспериментируйте с моделью и данными, попробуйте выбить наибольшее качество.

Сдайте в ЛМС наибольший accuracy, который у вас получился.